In [4]:
import numpy as np
import pandas as pd
import torch
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import SimpleImputer
from torch_geometric.data import Data
import os

In [5]:
# Load hospital training datasets
A_train = pd.read_csv("../data/processed/hospital_A/train.csv")
B_train = pd.read_csv("../data/processed/hospital_B/train.csv")
C_train = pd.read_csv("../data/processed/hospital_C/train.csv")

In [6]:
# Split Features and Labels
def split_features_labels(df):
    X = df.drop("classification", axis=1)
    y = df["classification"]
    return X, y

XA, yA = split_features_labels(A_train)
XB, yB = split_features_labels(B_train)
XC, yC = split_features_labels(C_train)

Graph Design
Nodes  : Each patient = 1 node
Edges  : K-Nearest Neighbors (clinical similarity)

In [7]:
# Graph Construction Function
def build_graph(X, y, k=5):
    """
    X: pandas DataFrame or numpy array (features)
    y: pandas Series or numpy array (labels)
    """

    # Convert to NumPy arrays
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    # Handle missing values
    imputer = SimpleImputer(strategy="median")
    X = imputer.fit_transform(X)

    # KNN-based graph
    n_samples = X.shape[0]
    k = min(k, n_samples - 1)

    knn = NearestNeighbors(n_neighbors=k, metric="euclidean")
    knn.fit(X)
    _, indices = knn.kneighbors(X)

    # Build edge index
    edge_index = []
    for i in range(n_samples):
        for j in indices[i]:
            if i != j:
                edge_index.append([i, j])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # Create PyTorch Geometric Data object
    graph = Data(
        x=torch.tensor(X, dtype=torch.float),
        y=torch.tensor(y, dtype=torch.long),
        edge_index=edge_index
    )

    return graph

In [8]:
# Build Graphs for Each Hospital
graph_A = build_graph(XA, yA, k=5)
graph_B = build_graph(XB, yB, k=5)
graph_C = build_graph(XC, yC, k=5)

print(graph_A)
print(graph_B)
print(graph_C)

Data(x=[320, 14], edge_index=[2, 1280], y=[320])
Data(x=[176, 14], edge_index=[2, 704], y=[176])
Data(x=[144, 14], edge_index=[2, 576], y=[144])


d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: [ 5  6  7  8 18 19 20 21 22 23]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: [ 5  6  7  8 18 19 20 21 22 23]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: [ 5  6  7  8 18 19 20 21 22 23]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [9]:
# Save Graphs (for reuse in training)
os.makedirs("../data/graph", exist_ok=True)

torch.save(graph_A, "../data/graph/graph_A.pt")
torch.save(graph_B, "../data/graph/graph_B.pt")
torch.save(graph_C, "../data/graph/graph_C.pt")